# Day 13：进阶语法与异步入门（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 2 周**：第 2 周：面向对象、工程化与命令行 AI 助手实战  
> **GitLab 工单**：`STARMIND-013` ｜ **开发分支**：`feat/day13-decorators-generators-asyncio`  
> **配套 HTML5 课件**：`courseware/day13/index.html` ｜ **独立版本快照**：`versions/day13/`

---

## 🎙️ 老陈开场旁白

今天（STARMIND-013）咱们给代码做一次「从新手脚本到工程组件」的脱胎换骨升级！
回想昨天调大模型 API 遇到的三个真实痛点：1）万一遇到 429 限流或短暂网络抖动，总不能每次都在业务代码里手写 `for` 循环重试吧？用**装饰器** `@retry_with_backoff` 一行搞定！
2）大模型生成几百字回答时用户不想傻等 10 秒，怎么像 ChatGPT 那样一个字一个字往外吐？靠 **`yield` 生成器**！
3）运营要批量让 AI 总结 20 篇文档，同步串行一篇 2 秒要等 40 秒，用 **`asyncio` 异步并发** + `Semaphore` 限流 + `wait_for` 超时控制，3 秒全搞定！
再加上 `python-dotenv` 安全管理 `.env` 里的密钥，今天学完你写出来的 Python 代码一眼就能看出是大厂工程师水准！

### 承上启下
- **昨日基线**：昨天咱们成功封装了 OpenAI 兼容 HTTP 客户端并跑通了本地 Mock 降级命令行问答，还用 `httpx.MockTransport` 覆盖了 401/429 异常测试。
- **明日预告**：明天（Day 14）就是第一阶段收官大考！咱们要把前 13 天的全部内功融会贯通，独立交付阶段考核项目一——**命令行多轮对话 AI 助手（带上下文记忆、JSON 持久化与 `/clear` `/save` `/exit` 指令）**！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day13'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-013）

随着星辰智脑开始对接外部大模型接口与批量文档分析任务，团队需要一套通用的底层可靠性与性能组件：避免密钥泄露、自动应对接口偶发抖动、支持流式分块输出以及多任务异步并发提速。

**今日迭代目标：**
- 掌握 `functools.wraps` 编写带参数的指数退避重试装饰器与耗时统计装饰器。
- 掌握 `yield` 生成器实现低内存流式文本分块输出，配合 `typing` 清晰标注类型。
- 掌握 `python-dotenv` 安全读取 `.env` 配置，以及 `asyncio.gather`、`asyncio.wait_for`、`asyncio.Semaphore` 实现带超时与并发上限的异步调用。

**验收标准：**
- 重试装饰器能在函数前 N-1 次抛异常、第 N 次成功时自动恢复，超过上限后准确抛出原异常。
- `.env` 配置加载器能读取本地 `.env` 文件且在文件缺失时安全回退到默认配置。
- `yield` 流式生成器返回标准 `Generator` 对象，逐块输出不一次性占用多余内存。
- 异步批量调度器能并发执行多个任务（总耗时远小于串行总和），并对超时的单个任务进行 `asyncio.TimeoutError` 隔离降级。

---

## 📚 课堂核心技术笔记

### 1. 1. 装饰器（@decorator）与 functools.wraps

装饰器本质是「接收一个函数、返回一个增强版新函数」的高阶函数。写带参数的装饰器（如 `@retry_with_backoff(max_retries=3)`）需要三层嵌套：`工厂层(参数) -> 装饰器层(func) -> 包装执行层(*args, **kwargs)`。
务必在最内层 `wrapper` 上加 `@functools.wraps(func)`，否则原函数的 `__name__` 和 `__doc__` 会被覆盖成 `wrapper`，排查日志和生成文档时会一团糟。

> **⚠️ 老陈避坑**：重试装饰器不要裸抓 `Exception` 后无脑重试所有错误——像参数写错的 `ValueError` 或 401 鉴权失败重试一百次也没用，应通过 `exceptions=(...)` 参数精准指定可重试的异常类型。

### 2. 2. 生成器（yield）与惰性求值

普通函数 `return` 会一次性把整个列表算完塞进内存；含 `yield` 的函数调用后返回一个生成器对象，每次 `next()` 或 `for` 循环才执行到下一个 `yield` 暂停并吐出一个元素。
在 AI 开发中，大模型流式响应（SSE）和 GB 级语料逐行清洗几乎全部依赖生成器。

> **⚠️ 老陈避坑**：生成器是**一次性消耗品**！遍历完一遍就空了，第二次 `for` 遍历什么都拿不到；如果确实需要多次复用，记得显式转为 `list()`（仅限小数据）。

### 3. 3. python-dotenv 与 asyncio 异步入门

- `python-dotenv`：把 `DEEPSEEK_API_KEY=...` 写在项目根目录的 `.env` 文件里（并将 `.env` 加入 `.gitignore` 永不提交），代码里用 `dotenv_values()` 或 `load_dotenv()` 读取。
- `asyncio`：`async def` 定义协程函数，`await` 挂起当前协程把控制权交还给事件循环；`asyncio.gather(*tasks)` 并发跑多个协程；`asyncio.wait_for(coro, timeout=...)` 做超时控制；`asyncio.Semaphore(n)` 限制最大并发数。

> **⚠️ 老陈避坑**：在 `async def` 协程里**绝对不要调用 `time.sleep()` 或同步 `requests.get()`**！它们会把整个事件循环线程卡死，所有并发瞬间退化成串行；请改用 `await asyncio.sleep()` 和 `httpx.AsyncClient`。


## 💻 实操步骤 1：步骤一：实现 .env 配置加载、指数退避重试装饰器与 yield 流式生成器

- **目标文件**：`versions/day13/src/day13_decorators_and_env.py`
- **老陈代码旁白**：先打造三件工程利器：`load_env_settings()` 安全加载 `.env`、`@retry_with_backoff` 实现指数退避重试、`stream_text_chunks()` 用 `yield` 模拟大模型流式分块吐字。


In [ ]:
"""
STARMIND-013 模块一：dotenv 配置加载、指数退避重试装饰器与 yield 流式生成器。
演示 python-dotenv、functools.wraps、typing 类型注解与生成器惰性求值。
"""

import functools
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Generator, Optional, Tuple, Type, TypeVar

from dotenv import dotenv_values

T = TypeVar("T")


@dataclass
class EnvSettings:
    """从 .env 安全加载的应用配置。"""

    model_name: str = "deepseek-chat"
    api_key: str = ""
    timeout_sec: float = 15.0
    max_retries: int = 3


def load_env_settings(env_path: Optional[Path] = None) -> EnvSettings:
    """
    安全解析 .env 文件并返回强类型 EnvSettings。
    文件不存在时平滑回退到默认配置，不污染全局 os.environ。
    """
    raw: dict[str, Optional[str]] = {}
    if env_path is not None and env_path.exists():
        raw = dict(dotenv_values(env_path))

    return EnvSettings(
        model_name=(raw.get("STARMIND_MODEL") or "deepseek-chat").strip(),
        api_key=(raw.get("STARMIND_API_KEY") or "").strip(),
        timeout_sec=float(raw.get("STARMIND_TIMEOUT") or 15.0),
        max_retries=int(raw.get("STARMIND_MAX_RETRIES") or 3),
    )


def retry_with_backoff(
    max_retries: int = 3,
    base_delay: float = 0.01,
    exceptions: Tuple[Type[BaseException], ...] = (RuntimeError, TimeoutError),
) -> Callable[[Callable[..., T]], Callable[..., T]]:
    """
    指数退避重试装饰器。
    当被装饰函数抛出指定 exceptions 时，按 base_delay * 2^attempt 休眠后重试。
    """
    if max_retries < 1:
        raise ValueError("max_retries 必须 >= 1")

    def decorator(func: Callable[..., T]) -> Callable[..., T]:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> T:
            last_exc: Optional[BaseException] = None
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    last_exc = exc
                    if attempt == max_retries - 1:
                        break
                    sleep_sec = base_delay * (2 ** attempt)
                    if sleep_sec > 0:
                        time.sleep(sleep_sec)
            assert last_exc is not None
            raise last_exc

        return wrapper

    return decorator


def stream_text_chunks(text: str, chunk_size: int = 4) -> Generator[str, None, None]:
    """使用 yield 按固定块大小惰性吐出文本，模拟大模型流式输出。"""
    if chunk_size <= 0:
        raise ValueError("chunk_size 必须为正整数")
    for idx in range(0, len(text), chunk_size):
        yield text[idx : idx + chunk_size]


if __name__ == "__main__":
    # 1. 演示重试装饰器：前两次模拟接口抖动，第三次成功
    state = {"calls": 0}

    @retry_with_backoff(max_retries=3, base_delay=0.005, exceptions=(RuntimeError,))
    def flaky_api_call(prompt: str) -> str:
        state["calls"] += 1
        if state["calls"] < 3:
            raise RuntimeError(f"第 {state['calls']} 次调用触发临时限流")
        return f"第 {state['calls']} 次成功返回：{prompt}"

    print(flaky_api_call("总结星辰数科年假制度"))

    # 2. 演示 yield 流式分块输出
    chunks = list(stream_text_chunks("星辰智脑正在流式生成回答！", chunk_size=4))
    print("流式分块结果:", chunks)


### 🔍 步骤 1 代码深度解析

1. `dotenv_values(env_path)` 只读取指定 `.env` 文件并返回字典，比全局修改环境变量的 `load_dotenv()` 更干净、更易于并发和测试隔离。
2. `@retry_with_backoff` 保留了原函数元信息（`@functools.wraps`），并且仅重试 `exceptions` 元组中声明的可恢复异常。
3. `stream_text_chunks` 的返回类型标注为 `Generator[str, None, None]`，配合 `yield` 实现按需产出，零多余内存拷贝。

**预期运行输出参考：**
```text
第 3 次成功返回：总结星辰数科年假制度
流式分块结果: ['星辰智脑', '正在流式', '生成回答', '！']
```


## 💻 实操步骤 2：步骤二：编写基于 asyncio 的并发大模型批量调用与超时控制引擎

- **目标文件**：`versions/day13/src/day13_async_llm_runner.py`
- **老陈代码旁白**：接下来咱们用 `asyncio` 写一个批量并发调用引擎：用 `asyncio.Semaphore` 限制最大并发度，用 `asyncio.wait_for` 给每个请求戴上超时安全帽，单个超时绝不拖垮整批任务！


In [ ]:
"""
STARMIND-013 模块二：asyncio 异步并发大模型调用与超时控制引擎。
演示 async/await、asyncio.wait_for 超时兜底、asyncio.Semaphore 并发限流与 asyncio.gather。
"""

import asyncio
import time
from typing import Any, Dict, List


async def async_mock_llm_call(prompt: str, delay_sec: float = 0.04) -> Dict[str, Any]:
    """模拟异步大模型 I/O 推理请求（非阻塞等待）。"""
    await asyncio.sleep(delay_sec)
    return {
        "prompt": prompt,
        "status": "ok",
        "answer": f"[异步回复] 已处理：{prompt}",
    }


async def invoke_with_timeout(
    prompt: str,
    delay_sec: float,
    timeout_sec: float,
) -> Dict[str, Any]:
    """为单次异步大模型调用增加超时保护，超时后优雅降级而不抛出未捕获异常。"""
    try:
        return await asyncio.wait_for(
            async_mock_llm_call(prompt, delay_sec=delay_sec),
            timeout=timeout_sec,
        )
    except asyncio.TimeoutError:
        return {
            "prompt": prompt,
            "status": "timeout",
            "answer": f"请求超时（>{timeout_sec}s），已触发降级兜底。",
        }


async def batch_ask_concurrently(
    prompts: List[str],
    delay_sec: float = 0.04,
    timeout_sec: float = 0.2,
    max_concurrency: int = 5,
) -> List[Dict[str, Any]]:
    """
    带并发上限控制的异步批量提问调度器。
    通过 asyncio.Semaphore 防止瞬时并发过高打爆上游 API 限流。
    """
    if max_concurrency < 1:
        raise ValueError("max_concurrency 必须 >= 1")

    semaphore = asyncio.Semaphore(max_concurrency)

    async def _bounded_task(p: str) -> Dict[str, Any]:
        async with semaphore:
            return await invoke_with_timeout(
                prompt=p,
                delay_sec=delay_sec,
                timeout_sec=timeout_sec,
            )

    tasks = [_bounded_task(p) for p in prompts]
    return list(await asyncio.gather(*tasks))


if __name__ == "__main__":
    questions = [f"批量问题 #{i}" for i in range(1, 6)]
    t0 = time.perf_counter()
    results = asyncio.run(
        batch_ask_concurrently(questions, delay_sec=0.05, timeout_sec=0.3, max_concurrency=5)
    )
    elapsed = (time.perf_counter() - t0) * 1000
    print(f"并发完成 {len(results)} 个任务（单个 50ms），总耗时仅 {elapsed:.1f}ms")
    print("首条结果:", results[0])


### 🔍 步骤 2 代码深度解析

1. `asyncio.Semaphore(max_concurrency)` 是生产环境必备保护阀：哪怕一次性丢进来 1000 篇文档，同一时刻也只会放行 `max_concurrency` 个请求去调 API，绝不把自己打进 429 黑名单。
2. `asyncio.wait_for()` 在协程超过 `timeout_sec` 时会自动取消它并抛出 `asyncio.TimeoutError`，我们在函数内部接住并转成 `status='timeout'` 结构化结果，保证其他 99 个正常任务不受牵连。

**预期运行输出参考：**
```text
并发完成 5 个任务（单个 50ms），总耗时仅 52.3ms
首条结果: {'prompt': '批量问题 #1', 'status': 'ok', 'answer': '[异步回复] 已处理：批量问题 #1'}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day13/tests/test_day13.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-013 自动化单元测试。"""

import asyncio
import inspect
import sys
import time
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day13_decorators_and_env import (
    load_env_settings,
    retry_with_backoff,
    stream_text_chunks,
)
from day13_async_llm_runner import batch_ask_concurrently, invoke_with_timeout


def test_load_env_settings_with_and_without_file(tmp_path: Path):
    # 缺失 .env 文件应平滑返回默认值
    default_cfg = load_env_settings(tmp_path / "missing.env")
    assert default_cfg.model_name == "deepseek-chat"
    assert default_cfg.api_key == ""

    # 存在 .env 文件时正确解析
    env_file = tmp_path / ".env"
    env_file.write_text(
        "STARMIND_MODEL=qwen-plus\nSTARMIND_API_KEY=sk-local-123\nSTARMIND_TIMEOUT=25.5\nSTARMIND_MAX_RETRIES=4\n",
        encoding="utf-8",
    )
    loaded = load_env_settings(env_file)
    assert loaded.model_name == "qwen-plus"
    assert loaded.api_key == "sk-local-123"
    assert loaded.timeout_sec == 25.5
    assert loaded.max_retries == 4


def test_retry_with_backoff_success_and_exhaustion():
    attempts = {"n": 0}

    @retry_with_backoff(max_retries=3, base_delay=0.001, exceptions=(RuntimeError,))
    def recover_on_third() -> str:
        """恢复测试文档字符串。"""
        attempts["n"] += 1
        if attempts["n"] < 3:
            raise RuntimeError("transient error")
        return "recovered"

    assert recover_on_third.__name__ == "recover_on_third"
    assert recover_on_third() == "recovered"
    assert attempts["n"] == 3

    @retry_with_backoff(max_retries=2, base_delay=0.001, exceptions=(RuntimeError,))
    def always_fail() -> str:
        raise RuntimeError("permanent error")

    with pytest.raises(RuntimeError, match="permanent error"):
        always_fail()


def test_stream_text_chunks_generator():
    gen = stream_text_chunks("星辰智脑AI助手", chunk_size=3)
    assert inspect.isgenerator(gen)
    parts = list(gen)
    assert parts == ["星辰智", "脑AI", "助手"]
    assert "".join(parts) == "星辰智脑AI助手"


def test_async_concurrent_batch_and_timeout():
    prompts = [f"Q{i}" for i in range(5)]
    start = time.perf_counter()
    results = asyncio.run(
        batch_ask_concurrently(prompts, delay_sec=0.04, timeout_sec=0.3, max_concurrency=5)
    )
    elapsed = time.perf_counter() - start
    assert len(results) == 5
    assert all(r["status"] == "ok" for r in results)
    # 5 个 40ms 任务并发执行，总耗时应明显小于串行的 200ms
    assert elapsed < 0.16

    # 验证单请求超时优雅降级
    timeout_res = asyncio.run(invoke_with_timeout("慢查询", delay_sec=0.1, timeout_sec=0.02))
    assert timeout_res["status"] == "timeout"
    assert "超时" in timeout_res["answer"]

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：带调用计数统计的装饰器 @count_calls

**题目要求：**
编写一个装饰器 `count_calls(func)`，使用 `functools.wraps` 保留函数元信息，并在包装函数上维护 `wrapper.call_count` 属性，每次调用自动加 1。

**思路提示：** 在定义完 `wrapper` 后初始化 `wrapper.call_count = 0`，在 `wrapper` 内部先 `wrapper.call_count += 1` 再调用原函数。


In [ ]:
"""Day 13 课后题 1：函数调用次数统计装饰器。"""

import functools
from typing import Any, Callable


def count_calls(func: Callable[..., Any]) -> Callable[..., Any]:
    """统计被装饰函数的累计调用次数，挂载在 wrapper.call_count 属性上。"""

    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        wrapper.call_count += 1  # type: ignore[attr-defined]
        return func(*args, **kwargs)

    wrapper.call_count = 0  # type: ignore[attr-defined]
    return wrapper


@count_calls
def greet(name: str) -> str:
    return f"你好，{name}"


if __name__ == "__main__":
    print(greet("林产品"))
    print(greet("阿杰"))
    print("累计调用次数:", greet.call_count)


**答案解析与评分标准：**

Python 中函数也是对象，可以直接挂载自定义属性（如 `call_count`），配合 `@functools.wraps` 既优雅又便于监控埋点。


### 作业 2（企业实战）：企业实战：基于 yield 的流式错误日志过滤生成器

**题目要求：**
编写生成器函数 `iter_error_lines(lines: Iterable[str], level: str = 'ERROR') -> Generator[str, None, None]`，惰性遍历海量日志行，仅 `yield` 去除首尾空白后包含 `[ERROR]`（或指定 level）的日志行。

**思路提示：** 在 `for line in lines:` 中判断 `f'[{level.upper()}]' in line`，命中则 `yield line.strip()`。


In [ ]:
"""Day 13 课后题 2：低内存流式错误日志过滤生成器。"""

from typing import Generator, Iterable


def iter_error_lines(
    lines: Iterable[str],
    level: str = "ERROR",
) -> Generator[str, None, None]:
    """惰性过滤包含指定日志级别的日志行，适合处理 GB 级大日志文件。"""
    tag = f"[{level.upper()}]"
    for line in lines:
        if tag in line:
            yield line.strip()


if __name__ == "__main__":
    sample_logs = [
        "2026-05-13 10:00:01 [INFO] 服务启动完成\n",
        "2026-05-13 10:00:05 [ERROR] 模型接口返回 429 限流\n",
        "2026-05-13 10:00:08 [WARN] 响应延迟偏高\n",
        "2026-05-13 10:00:12 [ERROR] 数据库连接超时\n",
    ]
    for err in iter_error_lines(sample_logs):
        print("捕获告警:", err)


**答案解析与评分标准：**

传入 `Iterable[str]`（既可以是打开的文件对象也可以是列表），配合 `yield` 实现 O(1) 恒定内存占用过滤大日志。


### 作业 3（架构进阶）：架构进阶：带并发限流的异步多节点健康探活器

**题目要求：**
编写异步函数 `async_health_check_all(services: list[tuple[str, float]], timeout_sec: float = 0.05, max_concurrency: int = 3) -> dict[str, str]`。其中每个元组为 `(服务名, 模拟响应耗时秒数)`，在超时内响应记为 `'healthy'`，超时记为 `'unhealthy'`，返回服务名到状态的映射字典。

**思路提示：** 结合 `asyncio.Semaphore(max_concurrency)`、`asyncio.wait_for(asyncio.sleep(delay), timeout=timeout_sec)` 和 `asyncio.gather`。


In [ ]:
"""Day 13 课后题 3：带并发限流与超时判定的异步服务探活器。"""

import asyncio
from typing import Dict, List, Tuple


async def async_health_check_all(
    services: List[Tuple[str, float]],
    timeout_sec: float = 0.05,
    max_concurrency: int = 3,
) -> Dict[str, str]:
    """并发探活多个微服务节点，按超时阈值标记 healthy / unhealthy。"""
    sem = asyncio.Semaphore(max_concurrency)

    async def _probe(name: str, latency: float) -> Tuple[str, str]:
        async with sem:
            try:
                await asyncio.wait_for(asyncio.sleep(latency), timeout=timeout_sec)
                return name, "healthy"
            except asyncio.TimeoutError:
                return name, "unhealthy"

    pairs = await asyncio.gather(*(_probe(n, d) for n, d in services))
    return dict(pairs)


if __name__ == "__main__":
    nodes = [
        ("deepseek-gateway", 0.01),
        ("qwen-gateway", 0.02),
        ("slow-legacy-rag", 0.12),
    ]
    print(asyncio.run(async_health_check_all(nodes, timeout_sec=0.05)))


**答案解析与评分标准：**

这道题直接对应生产级 AI 网关的定时探活后台协程：并发探测几十个上游模型节点，自动摘除超时慢节点。


---

## 🎯 老陈收尾复盘

今天这五把利器——**装饰器、`yield` 生成器、`typing` 类型注解、`python-dotenv`、`asyncio` 异步**——在后面第二阶段的 FastAPI、第三阶段的 RAG 批量向量化、第四阶段的 Agent 工具流里每天都会高频用到。
回去把指数退避重试和 `asyncio.gather` 再手敲一遍，明天咱们直接上第一阶段毕业考核项目！
